In [1]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer

with open("../rag/index/chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)
texts = [c["text"] for c in chunks]

In [2]:
# lahan, fast model (~90 MB). Pahilyanda internet varun download hoto
model = SentenceTransformer("all-MiniLM-L6-v2")

# normalize=True: vectors chi length 1, mhanun dot product = cosine similarity
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
print(embeddings.shape)          # (chunks, 384)

# save, mhanje parat encode karaychi garaj nahi
np.save("../rag/index/embeddings.npy", embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\bhumi\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bhumi\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

(190, 384)


In [3]:
def search(question, k=3):
    q = model.encode([question], normalize_embeddings=True)[0]   # prashna cha vector
    scores = embeddings @ q                                      # prattek chunk shi similarity
    top = scores.argsort()[::-1][:k]                             # sarvat javal che k
    return [(round(float(scores[i]), 3), chunks[i]) for i in top]

def show(question, k=3):
    print("\nQ:", question)
    for score, c in search(question, k):
        print(f"  {score} | {c['source']} | p{c['page']} | {c['text'][:100]}...")

In [4]:
for q in [
    "What is eGFR?",
    "Which tests check for kidney disease?",
    "What blood pressure target is recommended for CKD?",
    "What is albuminuria?",
    "What is specific gravity?",
    "Who won the cricket world cup?",
]:
    show(q)


Q: What is eGFR?
  0.597 | CKD Stages G1-G5 (NKF).pdf | p2 | rate (eGFR) and a urine test known as the urine albumin- creatinine ratio (uACR). Both tests are nee...
  0.517 | KDIGO 2024 Executive Summary.pdf | p13 | alues for common laboratory tests by eGFR so that clinicians may be able to better identify aberrant...
  0.493 | KDIGO 2024 Executive Summary.pdf | p15 | eared by the kidneys. Practice Point 4.2.2: For most people and clinical settings, validated eGFR eq...

Q: Which tests check for kidney disease?
  0.728 | CDC Testing.pdf | p2 | o perform other tests to check your kidneys. These could include monitoring your blood pressure, run...
  0.689 | CKD Stages G1-G5 (NKF).pdf | p2 | Helping make red blood cells Balancing important minerals in your body Helping maintain your blood p...
  0.673 | CDC Testing.pdf | p1 | tested. Urine tests One of the earliest signs of kidney disease is when protein leaks into your urin...

Q: What blood pressure target is recommended for CKD?
  0.6